In [4]:
## Import all the libraries
import torch
import torch.nn as nn
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

## Image Transformations
transform=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

## Load Dataset
train_dataset=ImageFolder(
    "Dataset/Train",
    transform=transform
)

## DataLoader
train_Loader=DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

## CNN Model
class CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.actv = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.flat = nn.Flatten()
        self.fc1 = nn.LazyLinear(128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.actv(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.actv(x)
        x = self.pool(x)
        x = self.flat(x)
        x = self.fc1(x)
        x = self.actv(x)
        x = self.fc2(x)
        return x

## Create Model
model = CNN(num_classes=len(train_dataset.classes))
model.train()

## Loss Function
Loss=nn.CrossEntropyLoss()

## Optimizer
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

## Training
epochs = 10
for epoch in range(epochs):
    running_loss = 0
    for images, labels in train_Loader:
        output = model(images)
        loss = Loss(output, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss : {running_loss / len(train_Loader):.4f}"
    )

## Save Model
torch.save(
    model.state_dict(),
    "saved_model.pth"
)

print("Model Saved Successfully!")




Epoch [1/10] Loss : 0.0000
Epoch [2/10] Loss : 0.0000
Epoch [3/10] Loss : 0.0000
Epoch [4/10] Loss : 0.0000
Epoch [5/10] Loss : 0.0000
Epoch [6/10] Loss : 0.0000
Epoch [7/10] Loss : 0.0000
Epoch [8/10] Loss : 0.0000
Epoch [9/10] Loss : 0.0000
Epoch [10/10] Loss : 0.0000
Model Saved Successfully!


In [9]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image


# ==========================
# CNN Model
# ==========================

class CNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.actv = nn.ReLU()
        self.pool = nn.MaxPool2d(2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.flat = nn.Flatten()
        self.fc1 = nn.LazyLinear(128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.conv1(x)
        x = self.actv(x)
        x = self.pool(x)
        x = self.conv2(x)
        x = self.actv(x)
        x = self.pool(x)
        x = self.flat(x)
        x = self.fc1(x)
        x = self.actv(x)
        x = self.fc2(x)
        return x


# ==========================
# Load Model
# ==========================

# The checkpoint was saved with a single output neuron (shape [1, 128]/[1]),
# so set num_classes=1 to match the saved state_dict.
model = CNN(num_classes=1)
model.load_state_dict(torch.load("saved_model.pth"))
model.eval()

transform = transforms.Compose([

    transforms.Resize((128,128)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )

])


# ==========================
# Load Image
# ==========================

image = Image.open("pexels-luiz-eduardo-pacheco-706192036-32155881.jpg")

image = image.convert("RGB")

image = transform(image)

image = image.unsqueeze(0)
# ==========================
# Prediction
# ==========================

with torch.no_grad():

    output = model(image)

    # For a single-logit binary model, apply sigmoid and threshold at 0.5
    prob = torch.sigmoid(output).squeeze().item()
    predicted = int(prob > 0.5)

    predicted = torch.argmax(output, dim=1)
    print(predicted)



tensor([0])


In [2]:
## Manually building the CNN architecture

import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3,padding=1)
        self.relu=nn.ReLU()
        self.pool=nn.MaxPool2d(2)
        self.conv2=nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1)
        self.avgpool=nn.AdaptiveAvgPool2d((1,1))
        self.flat=nn.Flatten()
        self.fc1=nn.Linear(64*32*32,128)
        self.fc2=nn.Linear(128,2) ## 2 is because it is a binary classification problem

    def forward(self,x):
        x=self.conv1(x)
        x=self.relu(x)
        x=self.pool(x)
        x=self.conv2(x)
        x=self.relu(x)
        x=self.pool(x)
        x=self.avgpool(x)
        x=self.flat(x)
        x=self.fc1(x)
        x=self.relu(x)
        x=self.fc2(x)
        return x
    



In [2]:
## Manually Building the Training Loop

import torch
import torch.nn as nn
from torchvision import transforms,datasets
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

## Transforming the Size of all images into equal ones.
transform=transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

## Images Loading and Assigning Labels
train_dataset=datasets.ImageFolder(
    "Dataset/Train",
    transform=transform
)

## Loading the Images with Labels
train_Loader=DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

## CNN Model
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1=nn.Conv2d(in_channels=3,out_channels=32,kernel_size=3,padding=1)
        self.relu=nn.ReLU()
        self.pool=nn.MaxPool2d(2)
        self.conv2=nn.Conv2d(in_channels=32,out_channels=64,kernel_size=3,padding=1)
        ##self.avgpool=nn.AdaptiveAvgPool2d((1,1))
        self.flat=nn.Flatten()
        self.fc1=nn.Linear(64*32*32,128)
        self.fc2=nn.Linear(128,2) ## 2 is because it is a binary classification problem

    def forward(self,x):
        x=self.conv1(x)
        x=self.relu(x)
        x=self.pool(x)
        x=self.conv2(x)
        x=self.relu(x)
        x=self.pool(x)
        ##x=self.avgpool(x)
        x=self.flat(x)
        x=self.fc1(x)
        x=self.relu(x)
        x=self.fc2(x)
        return x
model=CNN()

## Loss Function
criterion = nn.CrossEntropyLoss()

## Optimizer (updates the Weights)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

## Training Loop
epochs=10
for epoch in range(epochs):
    model.train()
    for images,labels in train_Loader:
        output=model(images)
        loss=criterion(output,labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}")

## Save Model
torch.save(
    model.state_dict(),
    "saved_model.pth"
)

print("Model Saved Successfully!")

Epoch 1: Loss = 0.1985
Epoch 2: Loss = 0.2556
Epoch 3: Loss = 0.0670
Epoch 4: Loss = 0.3925
Epoch 5: Loss = 0.0919
Epoch 6: Loss = 0.0013
Epoch 7: Loss = 0.0027
Epoch 8: Loss = 0.0002
Epoch 9: Loss = 0.0015
Epoch 10: Loss = 0.0002
Model Saved Successfully!


In [9]:
## Manually Building Prediction Flow
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image

# Transforming the Images
transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.5,0.5,0.5],
        std=[0.5,0.5,0.5]
    )
])

# Load image
image = Image.open("pexels-luiz-eduardo-pacheco-706192036-32155881.jpg").convert("RGB")

# Apply transforms
image = transform(image)

# Adding Batch Dimension
image = image.unsqueeze(0)

# Load the Trained Model
model = CNN()
model.load_state_dict(torch.load("saved_model.pth"))

# Putting into Prediction Mode
model.eval()

# Actual Prediction
with torch.no_grad():

    output = model(image)

    prediction = torch.argmax(output,dim=1)

Output=prediction.item()
if Output==0:
    print("Cat")
else:
    print("Dog")




Dog
